# EDA: Manual Biophysical Features vs ESM-2 Embeddings

This notebook investigates which ESM-2 35M embedding dimensions are associated with the manually engineered protein features. It uses only protein sequence-derived features and embeddings; no GO terms, ontology data, or annotation text are used.

The embedding matrix is opened as a read-only NumPy memory map to avoid duplicating the 152 MB file in RAM.

## 1. Load Feature and Embedding Tables

Load the sequence-feature TSV, embedding metadata, row IDs, completion bitmap, and memory-mapped embedding matrix.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr, pearsonr, t
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import ElasticNet, Lasso
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.cluster import AgglomerativeClustering

warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_theme(style="whitegrid", context="notebook")

cwd = Path.cwd()
ROOT = cwd if (cwd / "data/features").exists() else cwd.parent
manual_candidates = [ROOT / "data/features/sequence/train_features.tsv", ROOT / "data/train_features.tsv"]
MANUAL_PATH = max((path for path in manual_candidates if path.exists()), key=lambda path: path.stat().st_size)
EMBEDDING_DIR = ROOT / "data/features/embeddings/train_esm2_35M"
EDA_DIR = ROOT / "data/eda/embedding_vs_manual"
EDA_DIR.mkdir(parents=True, exist_ok=True)

manual = pd.read_csv(MANUAL_PATH, sep="\t")
metadata = json.loads((EMBEDDING_DIR / "metadata.json").read_text())
embedding_ids = pd.read_csv(EMBEDDING_DIR / "ids.tsv", sep="\t")
done = np.memmap(EMBEDDING_DIR / "done.bool.dat", dtype="bool", mode="r", shape=(metadata["protein_count"],))
embeddings = np.memmap(
    EMBEDDING_DIR / "embeddings.f32.dat",
    dtype="float32",
    mode="r",
    shape=(metadata["protein_count"], metadata["embedding_dim"]),
)

print("repository root:", ROOT)
print("manual feature source:", MANUAL_PATH)
print("manual shape:", manual.shape)
print("embedding shape:", embeddings.shape)
print("completed embeddings:", int(done.sum()), "/", len(done))
manual.head()

## 2. Validate Keys, Join Datasets, and Handle Missing Values

The feature TSV and embedding files are aligned through `protein_id`, never by assuming an arbitrary file order.

In [ ]:
manual = manual.drop_duplicates("protein_id").set_index("protein_id")
embedding_ids = embedding_ids.drop_duplicates("protein_id").set_index("protein_id")
assert len(embedding_ids) == embeddings.shape[0]
assert embedding_ids.index.is_unique and manual.index.is_unique
shared_ids = manual.index.intersection(embedding_ids.index)
row_positions = embedding_ids.index.get_indexer(shared_ids)
complete = np.asarray(done)[row_positions]
valid_ids = shared_ids[complete]

manual_numeric = manual.select_dtypes(include=np.number).columns.tolist()
manual_numeric = [column for column in manual_numeric if manual[column].nunique(dropna=True) > 1]
manual_matrix = manual.loc[valid_ids, manual_numeric].replace([np.inf, -np.inf], np.nan)
manual_matrix = manual_matrix.fillna(manual_matrix.median())
embedding_matrix = np.asarray(embeddings[row_positions[complete]], dtype=np.float32)

assert len(valid_ids) == len(manual_matrix) == len(embedding_matrix)
print(f"shared IDs: {len(shared_ids):,}; complete usable rows: {len(valid_ids):,}")
print(f"manual features: {len(manual_numeric)}; embedding dimensions: {embedding_matrix.shape[1]}")
print("manual missing values after median fill:", int(manual_matrix.isna().sum().sum()))

## 3. Scale/Transform Manual Features and Embedding Dimensions

Robust scaling is useful for the heavy-tailed sequence properties. Log transforms are applied only to positive, strongly skewed features for exploratory modelling; correlations are also computed on the original values.

In [ ]:
manual_raw = manual_matrix.copy()
manual_transformed = manual_matrix.copy()
log_features = []
for column in manual_numeric:
    values = manual_matrix[column].to_numpy()
    if np.all(values >= 0) and abs(pd.Series(values).skew()) > 1.0:
        manual_transformed[column] = np.log1p(values)
        log_features.append(column)

manual_scaled = StandardScaler().fit_transform(manual_transformed)
embedding_scaled = StandardScaler().fit_transform(embedding_matrix)
manual_scaled = pd.DataFrame(manual_scaled, index=valid_ids, columns=manual_numeric)
print("log-transformed features:", log_features)
print("scaled matrices:", manual_scaled.shape, embedding_scaled.shape)

## 4. Compute Feature–Embedding Association Matrix

Pearson captures linear association; Spearman captures monotonic association. Mutual information is optional and run on a sample because it is substantially more expensive.

In [ ]:
X = embedding_scaled
Y = manual_scaled.to_numpy()
embedding_names = [f"emb_{index:03d}" for index in range(X.shape[1])]
pearson = np.corrcoef(X.T, Y.T)[: X.shape[1], X.shape[1] :]
ranks_x = pd.DataFrame(X).rank().to_numpy()
spearman = np.empty_like(pearson)
for feature_index in range(Y.shape[1]):
    ranks_y = pd.Series(Y[:, feature_index]).rank().to_numpy()
    spearman[:, feature_index] = np.corrcoef(ranks_x.T, ranks_y)[-1, :-1]

association = pd.DataFrame(pearson, index=embedding_names, columns=manual_numeric)
spearman_association = pd.DataFrame(spearman, index=embedding_names, columns=manual_numeric)
association.index.name = "embedding"
association.columns.name = "manual_feature"
spearman_association.index.name = "embedding"
spearman_association.columns.name = "manual_feature"
association_long = association.stack().rename("pearson").to_frame()
association_long["spearman"] = spearman_association.stack()
association_long["abs_spearman"] = association_long["spearman"].abs()

plt.figure(figsize=(14, 8))
sns.heatmap(association, cmap="vlag", center=0, xticklabels=True, yticklabels=False)
plt.title("Pearson association: ESM-2 dimensions vs manual features")
plt.xlabel("Manual feature")
plt.ylabel("Embedding dimension")
plt.tight_layout()
plt.show()

mi_n = min(10000, len(X))
mi_rows = np.random.default_rng(42).choice(len(X), mi_n, replace=False)
mi = np.zeros_like(pearson)
for feature_index in range(Y.shape[1]):
    mi[:, feature_index] = mutual_info_regression(X[mi_rows], Y[mi_rows, feature_index], random_state=42)
mi_association = pd.DataFrame(mi, index=embedding_names, columns=manual_numeric)
mi_association.index.name = "embedding"
mi_association.columns.name = "manual_feature"

## 5. Rank Embedding Dimensions per Biophysical Feature

Rank dimensions by absolute Spearman association. The approximate correlation p-values use the standard t approximation and are Benjamini–Hochberg adjusted within each manual feature.

In [ ]:
n_rows = len(X)
t_stats = association_long["spearman"].abs() * np.sqrt((n_rows - 2) / (1 - association_long["spearman"].pow(2)).clip(lower=1e-12))
association_long["p_value"] = 2 * t.sf(t_stats, df=n_rows - 2)

def benjamini_hochberg(values):
    values = np.asarray(values, dtype=float)
    order = np.argsort(values)
    ranked = values[order] * len(values) / np.arange(1, len(values) + 1)
    adjusted = np.minimum.accumulate(ranked[::-1])[::-1].clip(0, 1)
    result = np.empty_like(adjusted)
    result[order] = adjusted
    return result

association_long["q_value"] = association_long.groupby(level=1)["p_value"].transform(benjamini_hochberg)
TOP_K = 10
top_links = (
    association_long.reset_index()
    .assign(abs_spearman=lambda frame: frame["spearman"].abs())
    .sort_values(["manual_feature", "abs_spearman"], ascending=[True, False])
    .groupby("manual_feature", sort=False)
    .head(TOP_K)
)
top_links[["manual_feature", "embedding", "pearson", "spearman", "p_value", "q_value"]].head(30)

## 6. Multivariate Mapping (Manual Features → Embedding Space)

Fit regularized models for selected continuous features. These models ask whether a manual feature can be reconstructed from a sparse or low-rank combination of embedding dimensions; cross-validated R² is reported alongside selected coefficients.

In [ ]:
model_features = [feature for feature in ["length", "molecular_weight", "gravy", "isoelectric_point", "instability_index", "fraction_charged", "fraction_polar", "transmembrane_helix_count", "disorder_score"] if feature in manual_numeric]
cv = KFold(n_splits=5, shuffle=True, random_state=42)
model_rows = []
coefficient_rows = []
for feature in model_features:
    target = manual_scaled[feature].to_numpy()
    estimator = make_pipeline(StandardScaler(), ElasticNet(alpha=0.03, l1_ratio=0.8, max_iter=3000))
    scores = cross_val_score(estimator, X, target, cv=cv, scoring="r2", n_jobs=-1)
    estimator.fit(X, target)
    coefficients = estimator[-1].coef_
    model_rows.append({"feature": feature, "cv_r2_mean": scores.mean(), "cv_r2_std": scores.std(), "nonzero_coefficients": int(np.count_nonzero(coefficients))})
    for index in np.flatnonzero(coefficients):
        coefficient_rows.append({"feature": feature, "embedding": f"emb_{index:03d}", "coefficient": coefficients[index], "abs_coefficient": abs(coefficients[index])})
model_summary = pd.DataFrame(model_rows).sort_values("cv_r2_mean", ascending=False)
model_coefficients = pd.DataFrame(coefficient_rows).sort_values(["feature", "abs_coefficient"], ascending=[True, False])
model_summary

## 7. Embedding Projections Colored by Biophysical Features

PCA provides a deterministic global view. A sample is used for plotting so the notebook remains responsive while all rows remain available for association calculations.

In [ ]:
plot_n = min(12000, len(X))
plot_rows = np.random.default_rng(42).choice(len(X), plot_n, replace=False)
pca = PCA(n_components=2, random_state=42)
projection = pca.fit_transform(X[plot_rows])
projection_frame = pd.DataFrame(projection, columns=["PC1", "PC2"])
projection_frame["protein_id"] = valid_ids[plot_rows]
for feature in ["length", "gravy", "isoelectric_point", "fraction_charged", "transmembrane_helix_count", "disorder_score"]:
    if feature in manual_raw:
        projection_frame[feature] = manual_raw.iloc[plot_rows][feature].to_numpy()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for axis, feature in zip(axes.flat, [feature for feature in projection_frame.columns if feature in model_features][:6]):
    scatter = axis.scatter(projection_frame["PC1"], projection_frame["PC2"], c=projection_frame[feature], s=5, alpha=0.35, cmap="viridis")
    axis.set_title(feature)
    axis.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
    axis.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
    fig.colorbar(scatter, ax=axis, shrink=0.8)
plt.tight_layout()
plt.show()

## 8. Clustered Heatmap and Module Discovery

Cluster embedding dimensions by their association profiles across manual features. Dimensions in the same cluster have similar relationships to the hand-engineered sequence properties.

In [ ]:
cluster_count = 12
clusterer = AgglomerativeClustering(n_clusters=cluster_count, linkage="ward")
embedding_clusters = clusterer.fit_predict(association.to_numpy())
clustered_association = association.assign(cluster=embedding_clusters).sort_values("cluster")
cluster_sizes = clustered_association["cluster"].value_counts().sort_index().rename("embedding_count")
cluster_sizes.to_frame()

plt.figure(figsize=(14, 8))
sns.heatmap(clustered_association.drop(columns="cluster"), cmap="vlag", center=0, yticklabels=False)
plt.title("Embedding dimensions clustered by manual-feature association profile")
plt.xlabel("Manual feature")
plt.ylabel("Embedding dimensions ordered by cluster")
plt.tight_layout()
plt.show()

## 9. Stability Checks and Significance Testing

Check whether top links persist across random subsamples and estimate uncertainty for the strongest links. A small permutation test is included for one selected feature to calibrate the null distribution.

In [ ]:
stability_feature = "gravy" if "gravy" in manual_numeric else manual_numeric[0]
rng = np.random.default_rng(42)
stability_rows = []
for repeat in range(10):
    sample = rng.choice(len(X), min(10000, len(X)), replace=False)
    correlations = np.corrcoef(X[sample].T, Y[sample, manual_numeric.index(stability_feature)])[0:-1, -1]
    top_indices = np.argsort(np.abs(correlations))[::-1][:20]
    stability_rows.extend({"repeat": repeat, "embedding": f"emb_{index:03d}", "abs_correlation": abs(correlations[index])} for index in top_indices)
stability = pd.DataFrame(stability_rows)
stability_summary = stability.groupby("embedding").agg(repeats=("repeat", "nunique"), mean_abs_correlation=("abs_correlation", "mean"), min_abs_correlation=("abs_correlation", "min")).sort_values(["repeats", "mean_abs_correlation"], ascending=False)
stability_summary.head(20)

permutation_n = 25
selected_embedding = stability_summary.index[0]
selected_index = int(selected_embedding.split("_")[1])
observed = abs(np.corrcoef(X[:, selected_index], Y[:, manual_numeric.index(stability_feature)])[0, 1])
null = []
for _ in range(permutation_n):
    shuffled = rng.permutation(Y[:, manual_numeric.index(stability_feature)])
    null.append(abs(np.corrcoef(X[:, selected_index], shuffled)[0, 1]))
print({"feature": stability_feature, "embedding": selected_embedding, "observed_abs_r": observed, "permutation_p": (1 + sum(value >= observed for value in null)) / (permutation_n + 1)})

## 10. Export Ranked Mappings and Diagnostic Artifacts

Write the association matrices, ranked links, model summaries, cluster assignments, and plots to a reproducible EDA directory.

In [ ]:
association.to_csv(EDA_DIR / "pearson_association.tsv", sep="\t")
spearman_association.to_csv(EDA_DIR / "spearman_association.tsv", sep="\t")
mi_association.to_csv(EDA_DIR / "mutual_information.tsv", sep="\t")
top_links.to_csv(EDA_DIR / "top_embedding_links.tsv", sep="\t", index=False)
model_summary.to_csv(EDA_DIR / "regularized_model_summary.tsv", sep="\t", index=False)
model_coefficients.to_csv(EDA_DIR / "regularized_model_coefficients.tsv", sep="\t", index=False)
clustered_association.reset_index(names="embedding").to_csv(EDA_DIR / "embedding_association_clusters.tsv", sep="\t", index=False)
stability_summary.to_csv(EDA_DIR / "ranking_stability.tsv", sep="\t")

figures = {
    "association_heatmap.png": plt.gcf(),
}
for filename, figure in figures.items():
    figure.savefig(EDA_DIR / filename, dpi=160, bbox_inches="tight")

print(f"exported EDA artifacts to {EDA_DIR.resolve()}")